<a href="https://colab.research.google.com/github/EgzonnOsmanaj/MesoAI/blob/main/GenAI_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install LlamaIndex core, readers, database, and processing utilities
!pip install llama-index llama-index-core llama-index-readers-file pypdf -q
!pip install llama-index-embeddings-huggingface -q
!pip install chromadb llama-index-vector-stores-chroma -q
!pip install sentence-transformers rank-bm25 -q
!pip install llama-index-llms-huggingface -q
!pip install transformers accelerate bitsandbytes -q

In [ ]:
import os
import re
import gc
import numpy as np
import requests
from typing import List
import torch
import json
import chromadb
from rank_bm25 import BM25Okapi
from google.colab import drive
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import BitsAndBytesConfig
from llama_index.core import Settings, SimpleDirectoryReader, VectorStoreIndex, StorageContext, PromptTemplate, get_response_synthesizer, SummaryIndex
from llama_index.core.node_parser import NodeParser
from llama_index.core.schema import BaseNode, TextNode, NodeWithScore
from llama_index.core.retrievers import BaseRetriever, VectorIndexRetriever
from llama_index.core.postprocessor.types import BaseNodePostprocessor
from llama_index.core.query_engine import RetrieverQueryEngine, RouterQueryEngine
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core.tools import QueryEngineTool
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.output_parsers import SelectionOutputParser

In [ ]:
# Create directory for the PDF
os.makedirs("docs", exist_ok=True)

# Define Model Parameters
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
RERANK_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
LOCAL_LLM_NAME = "microsoft/Phi-3-mini-4k-instruct"

# 1. Initialize Embedding Engine
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBED_MODEL_NAME)

# 2. Configure 4-Bit Quantization
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print("Downloading and loading LLM locally onto GPU...")
# 3. Initialize Local HuggingFace LLM
Settings.llm = HuggingFaceLLM(
    model_name=LOCAL_LLM_NAME,
    tokenizer_name=LOCAL_LLM_NAME,
    context_window=4096,          # Aligning with Phi-3-mini-4k-instruct's actual context window
    max_new_tokens=256,           # Further reduced generation limit to save memory
    generate_kwargs={"temperature": 0.1, "do_sample": False},
    model_kwargs={
        "quantization_config": quantization_config,
        "attn_implementation": "sdpa"  # CRITICAL FIX: Enables memory-efficient attention
    },
    device_map="cuda"
)
print("Local LLM successfully initialized with SDPA.")

In [ ]:
def clear_gpu():
    """Aggressively clears PyTorch CUDA memory and Python garbage."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect() # Clears inter-process communication memory

In [ ]:
from llama_index.core.node_parser import SentenceSplitter

ARTICLE_BOUNDARY = re.compile(
    r'(?=Article'
    r'|Recital'
    r'|ANNEX'
    r'|Chapter'
    r')',
    re.IGNORECASE
)

class LegalArticleNodeParser(NodeParser):
    """Custom LlamaIndex NodeParser with a safety fallback for massive legal clauses."""

    def _parse_nodes(self, nodes: List[BaseNode], show_progress: bool = False, **kwargs) -> List[BaseNode]:
        full_text = ""
        page_offsets = []

        for node in nodes:
            page_num = node.metadata.get("page_label", "Unknown")
            page_offsets.append((len(full_text), page_num))
            full_text += node.get_content() + "\n\n"

        def char_to_page(pos: int) -> str:
            page = page_offsets[0][1]
            for offset, pg in page_offsets:
                if offset <= pos:
                    page = pg
                else:
                    break
            return page

        # Split along legal boundaries
        raw_sections = ARTICLE_BOUNDARY.split(full_text)
        sections = [s.strip() for s in raw_sections if s.strip()]

        # SAFETY FALLBACK: LlamaIndex standard splitter to catch giant annexes
        fallback_splitter = SentenceSplitter(chunk_size=512, chunk_overlap=50)

        parsed_nodes = []
        search_pos = 0
        for j, section in enumerate(sections):
            idx = full_text.find(section[:80], search_pos)
            page = char_to_page(idx) if idx != -1 else "1"
            search_pos = max(0, idx)

            # If the article is insanely long, chunk it further
            if len(section) > 2000: # Approx 1000 tokens
                sub_chunks = fallback_splitter.split_text(section)
                for k, sub in enumerate(sub_chunks):
                    parsed_nodes.append(TextNode(
                        text=sub,
                        id_=f"eu_ai_act_article_{j}_part_{k}",
                        metadata={"title": "EU AI Act", "page_label": page, "article_index": j}
                    ))
            else:
                parsed_nodes.append(TextNode(
                    text=section,
                    id_=f"eu_ai_act_article_{j}",
                    metadata={"title": "EU AI Act", "page_label": page, "article_index": j}
                ))

        return parsed_nodes

In [ ]:
class CrossEncoderReranker(BaseNodePostprocessor):
    """Custom NodePostprocessor. Initializes model ONCE to save massive latency/VRAM."""

    def __init__(self):
        super().__init__()
        # Load the model into memory exactly once during initialization
        self._cross_encoder = CrossEncoder(RERANK_MODEL_NAME, device="cuda")

    def _postprocess_nodes(self, nodes: List[NodeWithScore], query_bundle=None) -> List[NodeWithScore]:
        nodes.sort(key=lambda x: x.score, reverse=True)
        return nodes[:5]

        query = query_bundle.query_str
        pairs = [(query, node.node.get_content()) for node in nodes]

        # Use the pre-loaded model
        scores = self._cross_encoder.predict(pairs)

        for node, score in zip(nodes, scores):
            node.score = float(score)

        nodes.sort(key=lambda x: x.score, reverse=True)
        return nodes[:5]

In [ ]:
def normalize_legal_query(query: str) -> str:
    """Normalizes informal shorthand into exact statutory terms before retrieval."""
    q = query.strip()
    q = re.sub(r"\bAI Act\b", "Regulation (EU) 2024/1689", q, flags=re.I)
    q = re.sub(r"\bhigh risk\b", "high-risk AI system", q, flags=re.I)
    return q

class AdvancedHierarchicalRetriever(BaseRetriever):
    """
    State-of-the-art Legal Retriever featuring:
    1. Query normalization to bridge lexical gaps.
    2. HyDE expansion for targeted semantic vector mapping.
    3. Sparse BM25 + Dense Hybrid RRF fusion.
    4. Hierarchical Context Expansion (Child-to-Parent swapping).
    """

    def __init__(self, index, parent_nodes: List[BaseNode], top_k=20):
        super().__init__()
        self._index = index
        self._parent_nodes = parent_nodes
        self._top_k = top_k

        # BM25 is map-indexed over the highly granular child text blocks
        tokenized_corpus = [node.get_content().lower().split() for node in parent_nodes]
        self._bm25 = BM25Okapi(tokenized_corpus)

    def _retrieve(self, query_bundle) -> List[NodeWithScore]:
        # Step 1: Pre-process and normalize user phrasing
        raw_query = query_bundle.query_str
        normalized_query = normalize_legal_query(raw_query)

        # Step 2: HyDE Transformation via standardized LLM settings
        # Step 2: HyDE Transformation (STRICT for Local LLMs)
        hyde_prompt = (
            f"You are a strict legal document generator. Write a short paragraph (3-5 sentences) "
            f"in the formal style of the EU AI Act that legalistically answers this query: '{normalized_query}'\n"
            f"CRITICAL INSTRUCTION: Output ONLY the legal text. Do NOT include any conversational "
            f"filler, greetings, or explanations. Start immediately with the legal text."
        )

        hypothetical_doc = str(Settings.llm.complete(hyde_prompt)).strip()

        # Step 3: Sparse Search via BM25 over exact phrase boundaries
        bm25_scores = self._bm25.get_scores(normalized_query.lower().split())
        top_bm25_idxs = np.argsort(bm25_scores)[::-1][:self._top_k]
        bm25_results = [self._parent_nodes[idx] for idx in top_bm25_idxs]

        # Step 4: Dense Vector Search using the HyDE expanded vector space
        vector_retriever = self._index.as_retriever(similarity_top_k=self._top_k)
        dense_results_with_score = vector_retriever.retrieve(hypothetical_doc)
        dense_results = [res.node for res in dense_results_with_score]

        # Step 5: Reciprocal Rank Fusion (RRF)
        fused_scores = {}
        node_map = {}

        for rank, node in enumerate(bm25_results, start=1):
            fused_scores[node.id_] = fused_scores.get(node.id_, 0.0) + 1.0 / (60 + rank)
            node_map[node.id_] = node

        for rank, node in enumerate(dense_results, start=1):
            fused_scores[node.id_] = fused_scores.get(node.id_, 0.0) + 1.0 / (60 + rank)
            node_map[node.id_] = node

        sorted_fused = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)[:20]

        return [NodeWithScore(node=node_map[nid], score=score) for nid, score in sorted_fused]


In [ ]:
# Define the strict legal QA template
strict_legal_prompt_str = (
    "You are an expert EU legal assistant. Your task is to provide highly precise, "
    "audit-ready answers based strictly on the provided legal text.\n"
    "Context information is below.\n"
    "---------------------\n"
    "{context_str}\n"
    "---------------------\n"
    "Given the context information and not prior knowledge, answer the query.\n"
    "If the answer is not fully supported by the context, strictly state: 'I do not have enough evidence.'\n"
    "CRITICAL INSTRUCTION: You MUST cite the specific page for every claim you make. "
    "Because the business context requires high verifiability, format citations at the end of sentences "
    "exactly like this: [Page X].\n\n"
    "Query: {query_str}\n"
    "Answer: "
)

legal_qa_template = PromptTemplate(strict_legal_prompt_str)

# Build the synthesizer with your custom template, ensuring it uses the current Settings.llm
custom_synthesizer = get_response_synthesizer(
    llm=Settings.llm, # Explicitly pass Settings.llm
    text_qa_template=legal_qa_template,
    response_mode="compact"
)

In [ ]:
# 1. Download PDF if not present
PDF_PATH = "docs/eu_ai_act.pdf"
if not os.path.exists(PDF_PATH):
    print("Downloading EU AI Act PDF...")
    url = "https://eur-lex.europa.eu/legal-content/EN/TXT/PDF/?uri=OJ:L_202401689"
    with open(PDF_PATH, "wb") as f:
        f.write(requests.get(url).content)

# 2. Document Loading and Ingestion Execution
print("Parsing PDF and building structural legal entities...")
raw_documents = SimpleDirectoryReader("docs").load_data()
legal_parser = LegalArticleNodeParser()
nodes = legal_parser.get_nodes_from_documents(raw_documents)
print(f"Generated {len(nodes)} isolated legal boundary nodes.")

# 3. Setup Persistent Storage Engine via ChromaDB
print("Initializing persistent Chroma client context...")
chroma_client = chromadb.PersistentClient(path="./chroma_persistent_db")

# Clear collection if it exists to allow for clean re-runs
try:
    chroma_client.delete_collection("production_legal_rag")
except Exception:
    pass

chroma_collection = chroma_client.create_collection("production_legal_rag")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)
storage_context = StorageContext.from_defaults(vector_store=vector_store)

# 4. Creating Index Structures
print("Embedding nodes and building index...")
index = VectorStoreIndex(nodes, storage_context=storage_context)


In [ ]:
# 5. Assembling components into the final engine architecture
print("Assembling LlamaIndex RetrieverQueryEngine...")
custom_retriever = AdvancedHierarchicalRetriever(index, nodes, top_k=7)
reranker_processor = CrossEncoderReranker()

production_engine = RetrieverQueryEngine(
    retriever=custom_retriever,
    response_synthesizer=custom_synthesizer,
    node_postprocessors=[reranker_processor]
)

# Clear dead memory from the GPU
gc.collect()
torch.cuda.empty_cache()

# 6. Pipeline Test Query Execution
test_query = "What obligations apply to providers of general-purpose AI models?"
print(f"\nEvaluating system accuracy for query: '{test_query}'\n")

response = production_engine.query(test_query)

print("=== GROUNDED GENERATION RESPONSE ===")
print(response.response)

print("\n=== VERIFIABLE LEGAL CITATION DATA ===")
for i, node in enumerate(response.source_nodes):
    print(f"Passage Reference [{i+1}] | Source Page: {node.metadata.get('page_label')}")

In [ ]:
print("Assembling Vanilla Baseline Engine...")
# 1. Standard dense retrieval (no BM25, no HyDE)
baseline_retriever = VectorIndexRetriever(
    index=index,
    similarity_top_k=5
)

# Create a response synthesizer for the baseline engine that uses the current Settings.llm
baseline_synthesizer = get_response_synthesizer(llm=Settings.llm, response_mode="compact")

# 2. Standard prompt (no strict legal formatting constraints)
baseline_engine = RetrieverQueryEngine(
    retriever=baseline_retriever,
    response_synthesizer=baseline_synthesizer # Explicitly pass the synthesizer
)

In [ ]:
import pandas as pd
import os

EVAL_DIR = "./eval_results"
os.makedirs(EVAL_DIR, exist_ok=True)

eval_data = [
    # --- Simple factual ---
    {
        "query": "What is the purpose of the AI Act?",
        "category": "simple_factual",
        "gold_answer": (
            "The AI Act aims to regulate AI systems according to risk and support "
            "trustworthy AI while protecting fundamental rights."
        ),
        "gold_pages": [1, 2, 3]
    },
    {
        "query": "Which AI practices are prohibited under the EU AI Act?",
        "category": "simple_factual",
        "gold_answer": (
            "Prohibited practices include subliminal manipulation, exploiting vulnerabilities, "
            "real-time remote biometric identification in public spaces (with exceptions), "
            "social scoring by public authorities, and emotion recognition in workplaces/education."
        ),
        "gold_pages": [51, 52, 53]
    },
    {
        "query": "What is the definition of an AI system under the Act?",
        "category": "simple_factual",
        "gold_answer": (
            "An AI system is a machine-based system designed to operate with varying levels of "
            "autonomy and that may exhibit adaptiveness after deployment."
        ),
        "gold_pages": [46]
    },
    # --- Deep context ---
    {
        "query": "What obligations apply to providers of general-purpose AI models?",
        "category": "deep_context",
        "gold_answer": (
            "Providers must maintain technical documentation, make information available to "
            "downstream integrators, implement a copyright compliance policy, and cooperate "
            "with the AI Office."
        ),
        "gold_pages": [84, 85, 86]
    },
    {
        "query": "What categories of AI systems are classified as high-risk?",
        "category": "deep_context",
        "gold_answer": (
            "High-risk systems include those used in critical infrastructure, education, "
            "employment, essential services, law enforcement, migration, and administration "
            "of justice."
        ),
        "gold_pages": [57, 58, 59]
    },
    {
        "query": "What are the obligations of deployers of high-risk AI systems?",
        "category": "deep_context",
        "gold_answer": (
            "Deployers must use high-risk AI systems in accordance with instructions, "
            "ensure human oversight, monitor operation, and inform providers of serious incidents."
        ),
        "gold_pages": [72, 73]
    },
    # --- Temporal / procedural ---
    {
        "query": "When do the transparency obligations for GPAI models start to apply?",
        "category": "temporal",
        "gold_answer": (
            "The GPAI provisions apply 12 months after entry into force, i.e. from August 2025."
        ),
        "gold_pages": [112, 113]
    },
    {
        "query": "What is the application timeline for the prohibited practices provisions?",
        "category": "temporal",
        "gold_answer": (
            "Prohibited practices provisions apply 6 months after entry into force, "
            "i.e. from February 2025."
        ),
        "gold_pages": [112, 113]
    },
    # --- Ambiguous / edge case ---
    {
        "query": (
            "A researcher develops an AI model exclusively for scientific research "
            "and does not place it on the market. Is the model covered by the AI Act?"
        ),
        "category": "edge_case",
        "gold_answer": (
            "No. AI systems developed exclusively for scientific research and development "
            "are excluded from the scope of the AI Act."
        ),
        "gold_pages": [46]
    },
    {
        "query": (
            "A provider releases a general-purpose AI model under an open-source licence "
            "and makes the weights publicly available. Do the GPAI documentation obligations apply?"
        ),
        "category": "edge_case",
        "gold_answer": (
            "Generally no — open-source GPAI models are exempt from the technical documentation "
            "and integrator-information obligations, unless the model presents systemic risk."
        ),
        "gold_pages": [85, 86]
    },
]

eval_df = pd.DataFrame(eval_data)
eval_df.to_csv(os.path.join(EVAL_DIR, "eval_questions.csv"), index=False)
print(f"Evaluation set: {len(eval_df)} queries across {eval_df['category'].nunique()} categories")
eval_df[["query", "category", "gold_pages"]]

In [ ]:
def calculate_retrieval_metrics(retrieved_pages, gold_pages):
    """Calculates Hit@5, Precision@5, and Recall@5."""
    ret_set = set([str(p) for p in retrieved_pages])
    gold_set = set([str(p) for p in gold_pages])

    hits = ret_set & gold_set
    hit_at_5 = 1 if len(hits) > 0 else 0
    precision = len(hits) / len(ret_set) if ret_set else 0.0
    recall = len(hits) / len(gold_set) if gold_set else 0.0

    return hit_at_5, precision, recall

results = []

for _, row in eval_df.iterrows():
    # 1. ALWAYS clear before starting a new complex query
    clear_gpu()

    query = row["query"]
    gold_pages = row["gold_pages"]
    print(f"\nEvaluating: '{query}'")

    # 2. Run Baseline
    base_res = baseline_engine.query(query)
    base_pages = [node.metadata.get("page_label") for node in base_res.source_nodes][:5]
    b_hit, b_prec, b_rec = calculate_retrieval_metrics(base_pages, gold_pages)

    # 3. Run Enhanced Production Engine
    enh_res = production_engine.query(query)
    enh_pages = [node.metadata.get("page_label") for node in enh_res.source_nodes][:5]
    e_hit, e_prec, e_rec = calculate_retrieval_metrics(enh_pages, gold_pages)

    results.append({
        "Query": query[:40] + "...",
        "Base Hit": b_hit, "Enh Hit": e_hit,
        "Base Prec": round(b_prec, 2), "Enh Prec": round(e_prec, 2),
        "Base Recall": round(b_rec, 2), "Enh Recall": round(e_rec, 2)
    })

    # 4. Delete the LlamaIndex response objects holding heavy tensors
    del base_res
    del enh_res
    # 5. Flush the GPU so the next loop starts with a clean slate
    clear_gpu()

results_df = pd.DataFrame(results)

print("\n" + "="*60)
print("FINAL PIPELINE EVALUATION COMPARISON")
print("="*60)
display(results_df)

print("\nAggregate Averages:")
print(results_df.mean(numeric_only=True).round(3))

In [ ]:
import json
import re # Ensure re is imported for regex operations

def extract_json_from_llm_output(text: str) -> dict:
    """
    Robustly extracts a JSON object from LLM generated text.
    Handles common issues like markdown wrappers and extraneous text before/after JSON.
    """
    # 1. Try to find JSON wrapped in ```json ... ``` markdown
    markdown_json_match = re.search(r'```json\n(.*?)\n```', text, re.DOTALL)
    if markdown_json_match:
        try:
            return json.loads(markdown_json_match.group(1))
        except json.JSONDecodeError:
            pass # Failed to parse even with markdown wrapper, try other methods

    # 2. Try to find a top-level JSON object using a non-greedy match for first { ... }
    # This is suitable for cases where JSON is embedded or followed by other text.
    json_match = re.search(r'(\{.*?})', text, re.DOTALL)
    if json_match:
        try:
            return json.loads(json_match.group(1))
        except json.JSONDecodeError:
            pass

    # 3. If the above failed, try a greedy match for { ... } and then attempt to clean.
    # This handles cases where extraneous text might follow valid JSON.
    greedy_json_match = re.search(r'\{.*\}', text, re.DOTALL)
    if greedy_json_match:
        candidate_json_str = greedy_json_match.group(0)
        try:
            # Attempt to parse the greedy match directly
            return json.loads(candidate_json_str)
        except json.JSONDecodeError as e:
            # If it fails, specifically check for "Extra data" error,
            # which means valid JSON was found, but then more characters followed.
            # In this case, we try to find the last valid JSON object by trimming.
            if "Extra data" in str(e):
                last_brace_index = candidate_json_str.rfind('}')
                if last_brace_index != -1:
                    clean_json = candidate_json_str[:last_brace_index + 1]
                    try:
                        return json.loads(clean_json)
                    except json.JSONDecodeError:
                        pass # Trimmed still not valid
            pass # Other JSONDecodeError, continue to fail

    raise ValueError("No valid JSON object could be extracted from the LLM response.")

def llm_judge_generation(query, gold_pages, generated_answer, retrieved_pages):
    """Uses the local LLM as an objective evaluator to score generation quality."""
    judge_prompt = f"""You are an objective evaluator for an EU AI Act legal QA system.
Score the GENERATED ANSWER on three dimensions, each on a scale of 1 to 3:
  - Correctness (1=wrong/contradicts context, 2=partially correct, 3=fully correct)
  - Grounding   (1=no page citations, 2=vague/incorrect citations, 3=specific accurate citations matching {retrieved_pages})
  - Completeness(1=key elements missing, 2=most present, 3=all key elements covered)

Question: {query}
Target Gold Pages: {gold_pages}
Generated Answer: {generated_answer}

Respond ONLY with valid JSON exactly matching this format:
{{"correctness": 3, "grounding": 3, "completeness": 3, "comment": "one short sentence explaining the score"}}
"""
    try:
        resp = str(Settings.llm.complete(judge_prompt))
        return extract_json_from_llm_output(resp)

    except Exception as e:
        return {"correctness": 0, "grounding": 0, "completeness": 0, "comment": f"Error: {e}"}

print("Running LLM-as-a-Judge Generation Evaluation...\n")
generation_scores = []

for _, row in eval_df.iterrows():
    clear_gpu()

    query = row["query"]
    gold_pages = row["gold_pages"]

    # Query the enhanced engine
    enh_res = production_engine.query(query)
    raw_llm_output = str(enh_res.response)

    # --- Start Cleaning LLM Output for Judge ---
    # The production_engine's LLM is sometimes repeating the prompt itself or instructions.
    # This confuses the judge LLM. We need to extract only the generated answer.
    cleaned_answer_text = raw_llm_output

    # 1. Try to find the actual answer after the last 'Answer:' marker from the prompt template
    answer_marker = "Answer:"
    if answer_marker in raw_llm_output:
        # Take everything after the *last* occurrence of "Answer:"
        potential_answer = raw_llm_output.rsplit(answer_marker, 1)[-1].strip()
        cleaned_answer_text = potential_answer

    # 2. Heuristic check: if the "answer" still starts with known prompt instructions,
    # it means the LLM has repeated the prompt or generated instructions instead of an answer.
    # In such cases, replace it with an explicit failure message for the judge.
    instruction_starters = [
        "CRITICAL INSTRUCTION: You MUST cite the specific page",
        "You are an expert EU legal assistant",
        "Context information is below.",
        "Given the context information and not prior knowledge",
        "Query:",
        "If the answer is not fully supported by the context"
    ]

    for starter in instruction_starters:
        if cleaned_answer_text.startswith(starter):
            # If the LLM output looks like a prompt or instructions,
            # consider it a failure to generate a proper answer.
            cleaned_answer_text = "The production engine LLM failed to generate a proper answer, instead outputting prompt instructions or structure."
            break # Stop checking once a match is found

    answer_text = cleaned_answer_text # Use this cleaned text for judging
    # --- End Cleaning LLM Output for Judge ---

    retrieved_pages = [node.metadata.get("page_label") for node in enh_res.source_nodes][:5]

    # Grade it using the local LLM
    score = llm_judge_generation(query, gold_pages, answer_text, retrieved_pages)

    generation_scores.append({
        "Query": query[:40] + "...",
        "Correctness": score.get("correctness", 0),
        "Grounding": score.get("grounding", 0),
        "Completeness": score.get("completeness", 0),
        "Judge Comment": score.get("comment", "")
    })

    # CRITICAL: Delete memory references and clear the cache
    del enh_res
    clear_gpu()

gen_results_df = pd.DataFrame(generation_scores)

print("="*60)
print("GENERATION QUALITY EVALUATION (Out of 3)")
print("="*60)
display(gen_results_df)
print("\nAverage Generation Scores:")
print(gen_results_df.mean(numeric_only=True).round(2))

In [ ]:
import re
import time
from llama_index.core.selectors import LLMSingleSelector
from llama_index.core.query_engine import RouterQueryEngine
from llama_index.core.tools import QueryEngineTool
from llama_index.core.output_parsers.selection import SelectionOutputParser

# 1. ROBUST OUTPUT PARSER
class RobustSelectionOutputParser(SelectionOutputParser):
    """Intercepts raw LLM text, cleans hallucinations, and isolates the JSON."""
    def parse(self, output: str):
        cleaned_output = output.replace("{{", "{").replace("}}", "}")
        match = re.search(r'\[\s*\{.*?\}\s*\]', cleaned_output, re.DOTALL)
        if match:
            cleaned_output = match.group(0)
        else:
            match_obj = re.search(r'\{.*?\}', cleaned_output, re.DOTALL)
            if match_obj:
                cleaned_output = "[" + match_obj.group(0) + "]"
        return super().parse(cleaned_output)

print("Configuring Agentic Routing Tools...")

# 2. DEFINE TOOLS
deep_search_tool = QueryEngineTool.from_defaults(
    query_engine=production_engine,
    name="deep_legal_search",
    description=(
        "USE THIS TOOL for answering specific, highly detailed questions about "
        "legal obligations, prohibitions, timelines, risk classifications, or constraints "
        "within the EU AI Act."
    ),
)

synopsis_tool = QueryEngineTool.from_defaults(
    query_engine=synopsis_engine,
    name="broad_summary_search",
    description=(
        "USE THIS TOOL for answering broad, high-level questions about the overall "
        "purpose, scope, general goals, or high-level summary of the EU AI Act. "
        "Do NOT use this for specific rules or penalty look-ups."
    ),
)

# 3. DEFINE RAW PROMPT STRING
router_prompt_str = """You are an autonomous routing agent. Your task is to select the single most appropriate tool from the list below to answer the user's question.

Here is the list of tools available to you (1 to {num_choices}):
---------------------
{context_list}
---------------------

The user's question is: "{query_str}"

Your ENTIRE response MUST be a valid JSON ARRAY containing exactly one object.
Output ONLY JSON. Do not add any conversational text.

Format Example:
[
  {{"choice": 1, "reason": "explanation"}}
]
"""

# 4. CRITICAL FIX: Pass the string and parser directly to from_defaults()
my_selector = LLMSingleSelector.from_defaults(
    llm=Settings.llm,
    prompt_template_str=router_prompt_str,
    output_parser=RobustSelectionOutputParser()
)

# 5. INITIALIZE THE ROUTER
router_engine = RouterQueryEngine(
    selector=my_selector,
    query_engine_tools=[deep_search_tool, synopsis_tool],
    verbose=True
)

# 6. EXECUTE TEST
print("\n" + "="*60)
print("TESTING AUTONOMOUS ROUTING")
print("="*60)

test_broad = "Give me a high-level overview of what the EU AI Act hopes to achieve."
print(f"\nQuery 1 (Broad): '{test_broad}'")
clear_gpu()
broad_response = router_engine.query(test_broad)
print(f"Answer: {str(broad_response)[:200]}...")

time.sleep(4)

test_specific = "What are the specific financial penalties for using prohibited AI practices?"
print(f"\nQuery 2 (Specific): '{test_specific}'")
clear_gpu()
specific_response = router_engine.query(test_specific)
print(f"Answer: {str(specific_response)[:200]}...")